# 01 · v0 BART model quality

The v0 model is a Bayesian Additive Regression Trees (BART) categorical model over three
contact features — **launch speed, launch angle, sprint speed** — that reproduces public
Baseball Savant xwOBA. This notebook loads `results/stage_{A,B,C}/` and shows how good it is.

**Headline:** player-season correlation with Savant ≈ **0.96** (100+ PA), calibration
error ≈ 0.04, and the quality saturates by ~50k training rows (Stage B) — Stage C on 100k
rows matches it within noise.

In [ ]:
# --- setup: find repo root, load helpers (run this first) ---
import sys, json
from pathlib import Path
import polars as pl
from IPython.display import Image, Markdown, display

pl.Config.set_tbl_rows(30)
here = Path.cwd()
ROOT = next((p for p in [here, *here.parents] if (p / "config.yaml").exists()), here)
sys.path.insert(0, str(ROOT))
RESULTS = ROOT / "results"

def jload(rel):
    return json.loads((RESULTS / rel).read_text())

def show_fig(rel, width=680):
    p = RESULTS / rel
    return Image(filename=str(p), width=width) if p.exists() else Markdown(f"_missing figure: {rel}_")

print("repo root:", ROOT)
print("results:  ", RESULTS, "(exists)" if RESULTS.exists() else "(MISSING)")

## Quality across the three fit stages\nStage A is a tiny **smoke run** (5k rows, expect low numbers); the real comparison is B (50k) → C (100k). C ≈ B means the model is data-saturated by ~50k rows. (ELPD isn't comparable across stages — each predicts a different number of events.)

In [ ]:
rows = []
for stage in ("stage_A", "stage_B", "stage_C"):
    m = jload(f"{stage}/metrics.json")
    rep, elpd = m["replication"], m["elpd"]
    rows.append({
        "stage": stage.replace("stage_", ""),
        "fit_rows": m.get("fit_rows"),
        "player_r_holdout": round(rep["player_r_holdout"], 4),
        "event_r_holdout": round(rep["event_r_holdout"], 4),
        "calib_ece": round(m["calibration"]["per_class"]["out"]["ece"], 4),
        "elpd_lppd": round(elpd["elpd_lppd"], 0),
        "elpd_se": round(elpd["elpd_se"], 0),
    })
pl.DataFrame(rows)

`elpd_lppd` for Stage C = **−80107 ± 244** over 122k holdout events — the anchor any
future model version must beat. Player r rises only marginally A→C while ELPD barely moves:
three features saturate quality early.

## Feature importance\nHow much of the model's explanatory power each feature carries (cumulative R²).

In [ ]:
vi = jload("stage_C/metrics.json")["variable_importance"]
pl.DataFrame({"feature": vi["feature_labels"],
              "r2_mean_cumulative": [round(x, 4) for x in vi["raw"]["r2_mean"]]})

## Figures (Stage C)

In [ ]:
show_fig("stage_C/figures/replication_player.png")

Player-season model xwOBA vs Savant — the 0.96 correlation. Tight diagonal = faithful replication.

In [ ]:
show_fig("stage_C/figures/calibration_reliability.png")

Predicted vs observed outcome frequency — near the diagonal means well-calibrated probabilities (ECE ≈ 0.04).

In [ ]:
show_fig("stage_C/figures/sprint_localization_curves.png")

Sprint-speed signal: weak/topped contact gains xwOBA with speed, barrels are flat — directionally correct (a fast runner beats out weak grounders, but you can't outrun a barrel).

In [ ]:
show_fig("stage_C/figures/variable_importance.png")

**Takeaway.** v0 is a faithful, well-calibrated xwOBA reconstruction from three
contact features. Next question: is it any more *accurate* than Savant itself? → notebook **02**.